In [2]:
import sys, os
sys.path.append('/content/drive/MyDrive/develop/Projects/quant-dev-git/src')
os.chdir('/content/drive/MyDrive/develop/Projects/quant-dev-git')
print(os.getcwd())

/content/drive/MyDrive/develop/Projects/quant-dev-git


In [12]:
from quant_dev.data.manager import DataManager
# 1. 準備數據
dm = DataManager()
ticker = "AAPL"
df = dm.get_or_fetch(ticker, timeframe="1d", days=500, force_download=True)
print(f"Success! Data shape: {df.shape}")

Success! Data shape: (342, 5)


In [14]:
from quant_dev.backtest.strategy import Strategy, StrategyConfig
import pandas as pd, numpy as np
# ----- 產生信號：SMA 交叉 -----
fast = 20
slow = 50
df[f'SMA_{fast}'] = df['Close'].rolling(fast).mean()
df[f'SMA_{slow}'] = df['Close'].rolling(slow).mean()

signal = pd.Series(0, index=df.index)
signal[df[f'SMA_{fast}'] > df[f'SMA_{slow}']] = 1   # 買入
signal[df[f'SMA_{fast}'] < df[f'SMA_{slow}']] = -1  # 賣出

# ----- 設定入市/出市目標價 -----
# 入市：用突破前 High（即係前一日 High）
entry_price = df['High'].shift(1)
# 出市：用跌破前 Low（即係前一日 Low）
exit_price = df['Low'].shift(1)
# ----- 設定 Config -----
config = StrategyConfig(
    ticker="AAPL",
    direction="buy",                    # 只做好倉
    mode="normal",                      # 正常模式（跟信號出入）
    entry_order_type="stop",            # 用 Stop Order 入市
    exit_order_type="limit",            # 用 Limit Order 出市
    gap_entry="open",                   # 跳空用開市價成交
    gap_exit="wait_close",              # 跳空等到收市先決定
    initial_capital=10000.0
)

# ----- 建立 Strategy -----
strat = Strategy(config, df)

# ----- 餵 data -----
strat.add_signal(signal)
strat.set_entry_price(entry_price)
strat.set_exit_price(exit_price)

# ----- 執行回測 -----
strat.run()
strat.df

,Open,High,Low,Close,Volume,SMA_20,SMA_50,position,entry,exit,signal,entry_price,exit_price
Date,,,,,,,,,,,,,
2025-03-24,219.838425,220.315897,217.431146,219.569839,44299500,NaN,NaN,0,NaN,NaN,0,NaN,NaN
2025-03-25,219.609642,222.922141,218.923266,222.573975,34493600,NaN,NaN,0,NaN,NaN,0,220.315897,217.431146
2025-03-26,222.335220,223.837293,219.311205,220.365631,34466100,NaN,NaN,0,NaN,NaN,0,222.922141,218.923266
2025-03-27,220.226370,223.807454,219.400730,222.673447,37094800,NaN,NaN,0,NaN,NaN,0,223.837293,219.311205
2025-03-28,220.504904,222.633656,216.535870,216.754715,39818600,NaN,NaN,0,NaN,NaN,0,223.807454,219.400730
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-28,340.029999,342.890015,335.600006,340.079987,51859000,319.9590,307.8296,1,NaN,NaN,1,339.570007,334.019989
2026-07-29,339.730011,344.570007,337.350006,338.190002,56090800,322.4005,308.5888,1,NaN,NaN,1,342.890015,335.600006
2026-07-30,333.100006,334.750000,329.589996,333.429993,74817800,324.3530,309.3006,1,NaN,NaN,1,344.570007,337.350006


In [5]:
df

,Open,High,Low,Close,Volume,sma_fast,sma_slow
Date,,,,,,,
2025-03-24,219.838440,220.315913,217.431161,219.569855,44299500,NaN,NaN
2025-03-25,219.609627,222.922126,218.923251,222.573959,34493600,NaN,NaN
2025-03-26,222.335220,223.837293,219.311205,220.365631,34466100,NaN,NaN
2025-03-27,220.226355,223.807439,219.400715,222.673431,37094800,NaN,NaN
2025-03-28,220.504904,222.633656,216.535870,216.754715,39818600,NaN,NaN
...,...,...,...,...,...,...,...
2026-07-28,340.029999,342.890015,335.600006,340.079987,51859000,319.9590,307.8296
2026-07-29,339.730011,344.570007,337.350006,338.190002,56090800,322.4005,308.5888
2026-07-30,333.100006,334.750000,329.589996,333.429993,74817800,324.3530,309.3006
